In [14]:
import sys
sys.path.append('..')
import torch
import soundfile as sf
import json
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import argparse

from encoder import FMEncoder, compute_spectrogram_cqt
from fm_ddsp import fm_renderer, make_mod_matrix
from nnAudio.features import CQT2010v2
from generate_dataset import generate_dataset
from train import train

%load_ext autoreload
%autoreload 2

# Stage 1 — Single carrier, no modulation
    # Fixed f0=440, algorithm with one carrier only, all mod_values=0, only carrier_weights and levels vary. The encoder just needs to learn "how loud is each operator." Trivial but establishes basic learning.
# Stage 2 — Single modulator, single carrier
    # Fixed f0=440, one modulator → one carrier, vary ratio of modulator and level. Encoder learns the relationship between ratio and sideband position.
# Stage 3 — Two operators, fixed algorithm
    # Still fixed f0, introduce a second modulator. Encoder learns to handle multiple simultaneous modulators.
# Stage 4 — Variable f0, fixed algorithm
    # Now vary f0 across MIDI range. Encoder learns pitch invariance — same timbre at different pitches should give same parameters.
# Stage 5 — Multiple algorithms
    # Introduce algorithm variety. Encoder learns to distinguish different routing topologies.
# Stage 6 — Full dataset
    # Everything varying simultaneously.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
def stage1_params():
    return {
        'f0': 440.0,
        'algorithm': 'algo_2',
        'mod_values': torch.zeros(7),
        'ratios': torch.ones(4),
        'levels': torch.rand(4) * 0.9 + 0.1,
        'carrier_weights': torch.tensor([0.0, 0.0, 0.0, 1.0])
    }

In [16]:
args = argparse.Namespace(
    n_examples = 10000,
    save_dir = '../data/stage1',
    Fs = 16000,
    duration = 1.0,
    seed = 127,
    overwrite=True
)
generate_dataset(args, param_fn = stage1_params)

Low pass filter created, time used = 0.0005 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0022 seconds
Generating example 00000/10000
Generating example 01000/10000
Generating example 02000/10000
Generating example 03000/10000
Generating example 04000/10000
Generating example 05000/10000
Generating example 06000/10000
Generating example 07000/10000
Generating example 08000/10000
Generating example 09000/10000
Generating example 09999/10000


In [17]:
# train stage 1
args = argparse.Namespace(
    data_dir = '../data/stage1',
    Fs = 16000,
    duration = 1.0,
    lr = 0.0001,
    n_epochs = 10
)
train(args)

Low pass filter created, time used = 0.0005 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0023 seconds


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/marcus/.local/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/marcus/.local/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
  File "/home/marcus/.local/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 401, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
  File "/home/marcus/.local/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 214, in collate
    return [
  File "/home/marcus/.local/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 215, in <listcomp>
    collate(samples, collate_fn_map=collate_fn_map)
  File "/home/marcus/.local/lib/python3.10/site-packages/torch/utils/data/_utils